In [ ]:
# 1. Install Unsloth and essential MLOps dependencies instantly
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft transformers bitsandbytes accelerate

import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

# 2. Configure the Base Model (Using 4-bit compression so it fits on the free GPU)
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# 3. Apply LoRA (Low-Rank Adaptation) Matrix patches to target behaviors
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

# ==============================================================================
# 4. Map Raw Data to an Explicit Prompt Layout
# ==============================================================================
# Define the standard structure the model should learn
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token # Critical: Prevents infinite gibberish text loops

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Apply data formatting and terminate with EOS token
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts }

# Load raw columns and explicitly apply our formatting transformation map
raw_dataset = load_dataset("gbharti/finance-alpaca", split = "train[:1000]")
dataset = raw_dataset.map(formatting_prompts_func, batched = True)


# ==============================================================================
# 5. Initialize the Speed-Optimized Trainer (Now with verified text data)
# ==============================================================================
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text", # Safely reads our new column
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 1,      # 💡 CHANGED FROM 2 TO 1 (Saves huge VRAM)
        gradient_accumulation_steps = 8,      # 💡 CHANGED FROM 4 TO 8 (Keeps mathematical weight stability)
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
    ),
)

# 6. Execute! Watch the math run live.
trainer_stats = trainer.train()

# 7. Push the lightweight trained behavioral adapter back to your Hugging Face Profile
# (Replace with your actual Hugging Face Username)
# model.push_to_hub("YOUR_USERNAME/llama3-financial-specialist", token="YOUR_HF_TOKEN")
print("Sprint Complete! Model adapter optimized successfully.")

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-1qbhj0bd/unsloth_7d2eb388e2644218bd1654e4fdbb8ef6
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-1qbhj0bd/unsloth_7d2eb388e2644218bd1654e4fdbb8ef6
  Resolved https://github.com/unslothai/unsloth.git to commit 677ec0cc20bf7cb4735385c51a22999a64839a83
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.6.9 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,2.788935
2,3.205847
3,2.850027
4,2.468747
5,2.885358
6,2.774313
7,2.773327
8,2.604824
9,2.282823
10,2.332497


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


Sprint Complete! Model adapter optimized successfully.


In [ ]:
# Compress your trained adapter files into a simple zip file
import os
os.system("zip -r financial_lora_adapter.zip outputs/checkpoint-60")

# Download it directly to your local computer's Downloads folder
from google.colab import files
files.download("financial_lora_adapter.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>